In [ ]:

# ── 표준 라이브러리 ──────────────────────────────────────────────
import os
import sys
import json
import math
import random
import argparse
import datetime
import time
from pathlib import Path
from copy import deepcopy
import glob
# ── DINO-main 경로 추가 ──────────────────────────────────────────
sys.path.insert(0, os.path.join(os.getcwd(), "DINO-main"))

# ── 수치 / 데이터 처리 ───────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# ── PyTorch ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ── DINO 모듈 ────────────────────────────────────────────────────
from util.slconfig import SLConfig
from util.misc import (
    nested_tensor_from_tensor_list,
    get_rank,
    is_main_process,
    save_on_master,
    MetricLogger,
    SmoothedValue,
)
from util import box_ops
from models import build_model
from datasets.coco import make_coco_transforms
from engine import train_one_epoch, evaluate

# ── 시각화 ────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ── 환경 정보 출력 ────────────────────────────────────────────────
print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA 사용 : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device    : {device}")


In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────
input_size   = 512
label_dir    = '../../data/HnE_cell_detect/total_data/labels/'
image_dir    = '../../data/HnE_cell_detect/total_data/images/'
screening_dir ='../../data/HnE_cell_detect/total_data/overlap/'
screening_files=sorted(glob.glob(os.path.join(screening_dir, '*.png')))
label_files=[f.replace('overlap', 'labels').replace('.png', '.json') for f in screening_files]


# 6 클래스 정의
class_names = {
    0: "Neutrophil",
    1: "Epithelial",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Eosinophil",
    5: "Connective tissue"
}
num_classes = len(class_names)

# ── 데이터 로드 ───────────────────────────────────────────────────
# JSON 형식: data_json["cordinates"] = [[class_id, y, x, h, w], ...]
from tqdm import tqdm

image_filenames = []
labels = []      # [{'boxes': ndarray(N,4) xyxy pixel, 'classes': ndarray(N,)}]

print("📂 Loading labels...")
for label_file in tqdm(label_files):
    with open(label_file) as f:
        data_json = json.load(f)

    img_path = os.path.join(image_dir, data_json['file_name'])
    if not os.path.exists(img_path):
        continue

    boxes, classes = [], []
    for coord in data_json.get("cordinates", []):
        if len(coord) < 5:
            continue
        class_id = int(coord[0]) - 1   # 1-based → 0-based
        y, x, h, w = coord[1], coord[2], coord[3], coord[4]
        if h > 50 or w > 50:           # 너무 큰 박스 제외
            continue
        x1, y1, x2, y2 = float(x), float(y), float(x + w), float(y + h)
        boxes.append([x1, y1, x2, y2])
        classes.append(class_id)

    if len(boxes) > 0:
        image_filenames.append(img_path)
        labels.append({
            'boxes':   np.array(boxes,   dtype=np.float32),   # (N,4) xyxy
            'classes': np.array(classes, dtype=np.int64)
        })

print(f"✅ {len(image_filenames)} images with labels loaded")

print("\n📷 Loading images...")
images = []
for img_path in tqdm(image_filenames):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append(img)

print(f"✅ {len(images)} images loaded  |  shape: {images[0].shape}")
print(f"   Label example – Boxes: {labels[0]['boxes'].shape}, Classes: {labels[0]['classes'].shape}")


# ── Dataset ───────────────────────────────────────────────────────
class DINOCellDataset(Dataset):
    """
    DINO용 H&E 세포 검출 데이터셋 (Rectangle bbox)
    - 입력 : 픽셀 단위 xyxy bbox
    - 출력 : DINO 포맷  boxes=[cx,cy,w,h] normalized [0,1], labels
    """
    def __init__(self, images, labels, img_size=512, augment=False):
        self.images   = images
        self.labels   = labels
        self.img_size = img_size
        self.augment  = augment

    def __len__(self):
        return len(self.images)

    # ── 색상 증강 (detail_p2pnet.ipynb 동일) ──────────────────────
    def apply_color_augmentation(self, image):
        # 1. Brightness ±20%
        if random.random() < 0.5:
            f = random.uniform(0.8, 1.2)
            image = np.clip(image * f, 0, 255).astype(np.uint8)
        # 2. Contrast ±20%
        if random.random() < 0.5:
            f = random.uniform(0.8, 1.2)
            mean = image.mean()
            image = np.clip((image - mean) * f + mean, 0, 255).astype(np.uint8)
        # 3. Hue shift ±10°
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 0] = np.clip(hsv[:, :, 0] + random.uniform(-10, 10), 0, 179)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        # 4. Saturation ±30%
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.7, 1.3), 0, 255)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        # 5. Gamma correction
        if random.random() < 0.3:
            inv_gamma = 1.0 / random.uniform(0.8, 1.2)
            table = np.array([((i / 255.0) ** inv_gamma) * 255
                              for i in range(256)]).astype(np.uint8)
            image = cv2.LUT(image, table)
        # 6. RGB channel shift ±10
        if random.random() < 0.3:
            for c in range(3):
                image[:, :, c] = np.clip(
                    image[:, :, c].astype(np.float32) + random.uniform(-10, 10), 0, 255
                ).astype(np.uint8)
        # 7. Gaussian noise
        if random.random() < 0.3:
            noise = np.random.normal(0, random.uniform(0, 5), image.shape)
            image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        # 8. Gaussian blur
        if random.random() < 0.2:
            ks = random.choice([3, 5])
            image = cv2.GaussianBlur(image, (ks, ks), 0)
        return image

    # ── 랜덤 크롭 / 패딩 ─────────────────────────────────────────
    def crop_padding_image(self, image):
        image = image.copy()
        h, w  = image.shape[:2]
        if min(h, w) > self.img_size:
            max_h = h - self.img_size
            max_w = w - self.img_size
            h1 = random.randint(0, max_h)
            w1 = random.randint(0, max_w)
            image = image[h1:h1 + self.img_size, w1:w1 + self.img_size]
        else:
            h1, w1 = 0, 0
            pad = np.ones((self.img_size, self.img_size, 3), dtype=np.uint8) * 255
            pad[:min(h, self.img_size), :min(w, self.img_size)] = \
                image[:min(h, self.img_size), :min(w, self.img_size)]
            image = pad
        return image, h1, w1

    # ── bbox 변환 유틸 ────────────────────────────────────────────
    @staticmethod
    def xyxy_to_cxcywh_norm(boxes_xyxy, img_size):
        """xyxy 픽셀 → cx,cy,w,h normalized [0,1]"""
        b  = boxes_xyxy.copy()
        cx = (b[:, 0] + b[:, 2]) / 2.0 / img_size
        cy = (b[:, 1] + b[:, 3]) / 2.0 / img_size
        bw = (b[:, 2] - b[:, 0])        / img_size
        bh = (b[:, 3] - b[:, 1])        / img_size
        return np.stack([cx, cy, bw, bh], axis=1)

    def __getitem__(self, index):
        original_image = self.images[index].copy()
        boxes_orig     = self.labels[index]['boxes'].copy()    # (N,4) xyxy pixel
        classes_orig   = self.labels[index]['classes'].copy()

        # 유효한 박스가 1개 이상 크롭 안에 들어올 때까지 반복
        crop_boxes, crop_classes = [], []
        while len(crop_boxes) < 1:
            image, h1, w1 = self.crop_padding_image(original_image)
            crop_boxes, crop_classes = [], []

            for i in range(len(classes_orig)):
                x1, y1, x2, y2 = boxes_orig[i]
                cx_orig = (x1 + x2) / 2.0
                cy_orig = (y1 + y2) / 2.0

                # 중심점이 크롭 영역 내에 있는 박스만 포함
                if not (w1 <= cx_orig <= w1 + self.img_size and
                        h1 <= cy_orig <= h1 + self.img_size):
                    continue

                nx1 = np.clip(x1 - w1, 0, self.img_size)
                ny1 = np.clip(y1 - h1, 0, self.img_size)
                nx2 = np.clip(x2 - w1, 0, self.img_size)
                ny2 = np.clip(y2 - h1, 0, self.img_size)

                if (nx2 - nx1) < 1 or (ny2 - ny1) < 1:
                    continue

                crop_boxes.append([nx1, ny1, nx2, ny2])
                crop_classes.append(classes_orig[i])

        boxes   = np.array(crop_boxes,   dtype=np.float32)   # (N,4) xyxy
        classes = np.array(crop_classes, dtype=np.int64)

        # ── 기하 증강 ───────────────────────────────────────────
        if self.augment:
            S = float(self.img_size)

            # Horizontal flip
            if random.random() < 0.5:
                image = np.fliplr(image).copy()
                old_x1, old_x2 = boxes[:, 0].copy(), boxes[:, 2].copy()
                boxes[:, 0] = S - old_x2
                boxes[:, 2] = S - old_x1

            # Vertical flip
            if random.random() < 0.5:
                image = np.flipud(image).copy()
                old_y1, old_y2 = boxes[:, 1].copy(), boxes[:, 3].copy()
                boxes[:, 1] = S - old_y2
                boxes[:, 3] = S - old_y1

            # 90° 회전 (1~3회)  xyxy → 90° CW: (x1,y1,x2,y2) → (y1, S-x2, y2, S-x1)
            if random.random() < 0.3:
                k = random.randint(1, 3)
                image = np.rot90(image, k).copy()
                for _ in range(k):
                    x1_, y1_, x2_, y2_ = (boxes[:, 0].copy(), boxes[:, 1].copy(),
                                          boxes[:, 2].copy(), boxes[:, 3].copy())
                    boxes[:, 0] = y1_
                    boxes[:, 1] = S - x2_
                    boxes[:, 2] = y2_
                    boxes[:, 3] = S - x1_

            # 색상 증강
            image = self.apply_color_augmentation(image)

        # ── DINO 포맷 변환: xyxy → cx,cy,w,h norm ───────────────
        boxes_norm = self.xyxy_to_cxcywh_norm(boxes, self.img_size)

        image = image.astype(np.float32) / 255.0
        image = image.transpose(2, 0, 1)   # HWC → CHW

        return (
            torch.from_numpy(image).float(),
            torch.from_numpy(boxes_norm).float(),   # (N,4) cx,cy,w,h norm
            torch.from_numpy(classes).long()         # (N,)
        )


# ── collate_fn (가변 길이 bbox 처리) ─────────────────────────────
def collate_fn_dino(batch):
    """DINO 포맷: targets = list of dict {'boxes':(N,4), 'labels':(N,)}"""
    images, targets = [], []
    for img, boxes, lbls in batch:
        images.append(img)
        targets.append({'boxes': boxes, 'labels': lbls})
    images = torch.stack(images, dim=0)
    return images, targets


# ── Train / Val split ────────────────────────────────────────────
from sklearn.model_selection import train_test_split

train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.1, random_state=242, shuffle=True
)

train_dataset = DINOCellDataset(train_images, train_labels, img_size=input_size, augment=True)
val_dataset   = DINOCellDataset(val_images,   val_labels,   img_size=input_size, augment=False)

print(f"\n📊 Dataset split:")
print(f"  Train : {len(train_dataset)} samples")
print(f"  Val   : {len(val_dataset)}   samples")

# ── 클래스 분포 및 WeightedRandomSampler ────────────────────────
# 클래스 불균형 해소: 희귀 클래스가 포함된 이미지를 더 자주 샘플링
total_counts = np.zeros(num_classes, dtype=np.float64)
for lbl in train_labels:
    for c in lbl['classes']:
        total_counts[c] += 1

print(f"\n📊 Train 클래스 분포:")
for i in range(num_classes):
    print(f"  [{i}] {class_names[i]:20s}: {int(total_counts[i]):6d}개")

# sqrt 역빈도 가중치: 72배 불균형 → 8.5배로 완화 (Epithelial 성능 보호)
# 1/count 사용 시 Eosinophil 이미지가 72배 과샘플링되어 Epithelial 붕괴 위험
cls_w = 1.0 / np.sqrt(total_counts + 1.0)

print(f"\n  상대 샘플링 비율 (Epithelial 기준):")
for i in range(num_classes):
    ratio = cls_w[i] / cls_w[1]   # Epithelial(idx=1) 대비
    print(f"  [{i}] {class_names[i]:20s}: {ratio:.1f}x")

# 이미지별 샘플링 가중치: 해당 이미지에 있는 가장 희귀한 클래스 기준
sample_weights = []
for lbl in train_labels:
    if len(lbl['classes']) == 0:
        sample_weights.append(1.0)
    else:
        sample_weights.append(float(max(cls_w[c] for c in lbl['classes'])))

sample_weights = torch.tensor(sample_weights, dtype=torch.double)
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

batch_size   = 4
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, sampler=sampler,   # shuffle=False (sampler가 대체)
    num_workers=4, collate_fn=collate_fn_dino, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn_dino, pin_memory=True
)

print(f"\n📦 DataLoaders ready:")
print(f"  Batch size    : {batch_size}")
print(f"  Train batches : {len(train_loader)}  (WeightedRandomSampler 적용)")
print(f"  Val batches   : {len(val_loader)}")
print(f"\n🎨 Augmentations (Train only):")
print(f"  Geometric : H-flip / V-flip / 90° rotation  (bbox 자동 변환)")
print(f"  Color     : Brightness, Contrast, Hue, Saturation, Gamma, RGB-shift, Noise, Blur")

# ── 샘플 확인 ─────────────────────────────────────────────────────
img_t, boxes_t, labels_t = train_dataset[0]
print(f"\n✅ Sample check:")
print(f"  Image  : {tuple(img_t.shape)}  dtype={img_t.dtype}")
print(f"  Boxes  : {tuple(boxes_t.shape)}  (cx,cy,w,h norm)  min={boxes_t.min():.3f}  max={boxes_t.max():.3f}")
print(f"  Labels : {tuple(labels_t.shape)}  unique={labels_t.unique().tolist()}")


In [ ]:

# ── 데이터셋 샘플 시각화 ─────────────────────────────────────────
def visualize_dataset_sample(dataset, index=0, title_prefix=""):
    img_t, boxes_t, labels_t = dataset[index]

    image  = img_t.numpy().transpose(1, 2, 0)          # CHW → HWC
    boxes  = boxes_t.numpy()                            # (N,4) cx,cy,w,h norm
    labels = labels_t.numpy()
    S      = image.shape[0]                             # img_size

    # cx,cy,w,h norm → x1,y1,x2,y2 픽셀
    cx, cy, bw, bh = boxes[:,0]*S, boxes[:,1]*S, boxes[:,2]*S, boxes[:,3]*S
    x1, y1 = cx - bw/2, cy - bh/2
    x2, y2 = cx + bw/2, cy + bh/2

    colors = ['red', 'limegreen', 'yellow', 'magenta', 'dodgerblue', 'orange']

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image)

    for i in range(len(boxes)):
        cls   = int(labels[i])
        color = colors[cls % len(colors)]
        rect  = patches.Rectangle(
            (x1[i], y1[i]), x2[i]-x1[i], y2[i]-y1[i],
            linewidth=1.5, edgecolor=color, facecolor='none', alpha=0.85
        )
        ax.add_patch(rect)

    legend_elements = [
        patches.Patch(color=colors[i], label=f'{class_names[i]} ({(labels==i).sum()})')
        for i in range(num_classes) if (labels == i).sum() > 0
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
              framealpha=0.7, edgecolor='white')
    ax.set_title(f"{title_prefix}Sample [{index}]  —  {len(boxes)} boxes", fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Train 샘플 1개
visualize_dataset_sample(train_dataset, index=0, title_prefix="[Train] ")


In [ ]:

# ── DINO 공식 COCOVisualizer.addtgt 방식 그대로 적용 ─────────────
# visualizer.py 의 addtgt 좌표변환 + PatchCollection 드로잉을 직접 사용
# (pycocotools import 오류 방지를 위해 inline 구현)
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

def dino_addtgt(ax, tgt):
    """
    DINO COCOVisualizer.addtgt 원본 로직 그대로.
    tgt: {'boxes': (N,4) cx,cy,w,h norm,
          'size' : tensor([H, W]),
          'box_label': list[str] (optional)}
    """
    H, W = tgt['size'].tolist()
    boxes_px = []
    polygons = []
    colors   = []

    for box in tgt['boxes'].cpu():
        # DINO 원본: box * [W, H, W, H]  →  cx_px, cy_px, w_px, h_px
        unnorm = box * torch.tensor([W, H, W, H])
        # cx,cy → top-left x,y
        unnorm[:2] -= unnorm[2:] / 2
        bx, by, bw, bh = unnorm.tolist()
        boxes_px.append([bx, by, bw, bh])

        poly = [[bx, by], [bx, by+bh], [bx+bw, by+bh], [bx+bw, by]]
        polygons.append(MplPolygon(np.array(poly).reshape(4, 2)))
        c = (np.random.random((1, 3)) * 0.6 + 0.4).tolist()[0]
        colors.append(c)

    # 반투명 채움 + 외곽선 (DINO 원본 방식)
    ax.add_collection(PatchCollection(polygons, facecolor=colors, linewidths=0, alpha=0.1))
    ax.add_collection(PatchCollection(polygons, facecolor='none', edgecolors=colors, linewidths=2))




# ── 배치 1개 추출 ─────────────────────────────────────────────────
batch_imgs, batch_targets = next(iter(train_loader))

n = len(batch_targets)
fig, axes = plt.subplots(1, n, figsize=(6*n, 7))
if n == 1:
    axes = [axes]

for i, (img_t, tgt) in enumerate(zip(batch_imgs, batch_targets)):
    ax = axes[i]

    # ── 이미지: CHW [0,1] → HWC (renorm 없음, 우리 이미지는 /255만 적용)
    img_np = np.clip(img_t.permute(1, 2, 0).numpy(), 0, 1)
    ax.imshow(img_np)

    # ── DINO addtgt 방식으로 박스 그리기
    tgt_viz = {
        'boxes':     tgt['boxes'],                                          # cx,cy,w,h norm
        'size':      torch.tensor([input_size, input_size], dtype=torch.float),  # [H, W]
        'box_label': [class_names[int(l)] for l in tgt['labels']],
    }
    dino_addtgt(ax, tgt_viz)

    ax.set_title(f'[{i}]  N={len(tgt["boxes"])}', fontsize=9)
    ax.axis('off')

plt.suptitle('DINO COCOVisualizer.addtgt — DataLoader 출력 그대로 시각화', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── 설정 로드 ─────────────────────────────────────────────────────
cfg = SLConfig.fromfile('DINO-main/config/DINO/DINO_4scale.py')

# H&E 세포 검출용 오버라이드
cfg.num_classes        = num_classes   # 6
cfg.dn_labelbook_size  = num_classes   # DN labelbook도 클래스 수에 맞춤
cfg.device             = str(device)

# (선택) 학습 하이퍼파라미터 조정
cfg.lr                 = 5e-5
cfg.lr_backbone        = 5e-5
cfg.lr_linear_proj_mult = 0.1
cfg.weight_decay       = 1e-4
cfg.clip_max_norm      = 0.1
cfg.lr_drop            = 40            # (CosineAnnealingLR에서는 미사용)
cfg.epochs             = 1000

# 세포 크기가 작으므로 num_queries 조정 (기본 900)
cfg.num_queries        = 2000

print("✅ Config loaded")
print(f"  num_classes    : {cfg.num_classes}")
print(f"  dn_labelbook   : {cfg.dn_labelbook_size}")
print(f"  num_queries    : {cfg.num_queries}")
print(f"  backbone       : {cfg.backbone}")
print(f"  use_dn         : {cfg.use_dn}")
print(f"  aux_loss       : {cfg.aux_loss}")


# ── 모델 빌드 ─────────────────────────────────────────────────────
# build_model → (DINO, SetCriterion, PostProcess)
model, criterion, postprocessors = build_model(cfg)
model     = model.to(device)
criterion = criterion.to(device)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ DINO model built")
print(f"  Total params     : {n_total:,}")
print(f"  Trainable params : {n_trainable:,}")


# ── Optimizer (3-group lr) ────────────────────────────────────────
# backbone < linear_proj < transformer  순으로 lr 차등 적용
from util.get_param_dicts import match_name_keywords

param_dicts = [
    {   # Transformer (기본 lr)
        "params": [
            p for n, p in model.named_parameters()
            if not match_name_keywords(n, cfg.lr_backbone_names)
            and not match_name_keywords(n, cfg.lr_linear_proj_names)
            and p.requires_grad
        ],
        "lr": cfg.lr,
    },
    {   # Backbone (낮은 lr — pretrained 가중치 보존)
        "params": [
            p for n, p in model.named_parameters()
            if match_name_keywords(n, cfg.lr_backbone_names)
            and p.requires_grad
        ],
        "lr": cfg.lr_backbone,
    },
    {   # Linear projection (reference_points, sampling_offsets — 매우 낮은 lr)
        "params": [
            p for n, p in model.named_parameters()
            if match_name_keywords(n, cfg.lr_linear_proj_names)
            and p.requires_grad
        ],
        "lr": cfg.lr * cfg.lr_linear_proj_mult,
    },
]

optimizer = torch.optim.AdamW(param_dicts, weight_decay=cfg.weight_decay)

print(f"\n✅ Optimizer: AdamW")
print(f"  Transformer lr   : {cfg.lr}")
print(f"  Backbone lr      : {cfg.lr_backbone}")
print(f"  Linear proj lr   : {cfg.lr * cfg.lr_linear_proj_mult}")
print(f"  Weight decay     : {cfg.weight_decay}")

# ── Scheduler ─────────────────────────────────────────────────────
# CosineAnnealingLR: 5e-5 → 2e-6 으로 부드럽게 감소 (1000 epoch)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=2e-6
)
print(f"\n✅ Scheduler: CosineAnnealingLR  (T_max={cfg.epochs}, eta_min=2e-6)")


# ── 저장 경로 ─────────────────────────────────────────────────────
save_dir = '../../model/HnE_cell_detection/dino/'
os.makedirs(save_dir, exist_ok=True)
print(f"\n📂 Checkpoint dir : {save_dir}")


# ── Loss 가중치 확인 ──────────────────────────────────────────────
print(f"\n✅ Criterion: SetCriterion (Hungarian matching)")
print(f"  cls_loss_coef  : {cfg.cls_loss_coef}")
print(f"  bbox_loss_coef : {cfg.bbox_loss_coef}")
print(f"  giou_loss_coef : {cfg.giou_loss_coef}")
print(f"  focal_alpha    : {cfg.focal_alpha}")
print(f"  aux_loss       : {cfg.aux_loss}  (decoder 각 층에서 loss 계산)")


In [ ]:
import numpy as np
from util.utils import ModelEma

# ── 학습 추적 변수 ─────────────────────────────────────────────────
train_losses  = []
val_map50s    = []
val_map5095s  = []
best_val_map  = 0.0
start_epoch   = 0

# ── AMP GradScaler ────────────────────────────────────────────────
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ── EMA ──────────────────────────────────────────────────────────
ema_model = ModelEma(model, decay=cfg.ema_decay) if cfg.use_ema else None
print(f"✅ AMP  : {torch.cuda.is_available()}")
print(f"✅ EMA  : {cfg.use_ema}" + (f"  (decay={cfg.ema_decay})" if cfg.use_ema else ""))


# ── mAP 유틸 ─────────────────────────────────────────────────────
def box_cxcywh_to_xyxy_pixel(boxes_norm, img_size):
    cx, cy, w, h = boxes_norm.unbind(-1)
    return torch.stack([
        (cx - w/2)*img_size, (cy - h/2)*img_size,
        (cx + w/2)*img_size, (cy + h/2)*img_size,
    ], dim=-1)

def box_iou_np(box, boxes):
    """box(4,) vs boxes(N,4) → IoU(N,)  xyxy numpy"""
    ix1 = np.maximum(box[0], boxes[:,0]); iy1 = np.maximum(box[1], boxes[:,1])
    ix2 = np.minimum(box[2], boxes[:,2]); iy2 = np.minimum(box[3], boxes[:,3])
    inter = np.maximum(ix2-ix1, 0) * np.maximum(iy2-iy1, 0)
    a1    = (box[2]-box[0])*(box[3]-box[1])
    a2    = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
    return inter / (a1 + a2 - inter + 1e-6)

def compute_ap(recall, precision):
    """Area under PR curve (VOC 2010+ style)"""
    mrec = np.concatenate(([0.], recall, [1.]))
    mpre = np.concatenate(([0.], precision, [0.]))
    for i in range(mpre.size-1, 0, -1):
        mpre[i-1] = max(mpre[i-1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx+1] - mrec[idx]) * mpre[idx+1]))

def compute_map(results, targets, img_size, n_cls,
                iou_thrs=None):
    """
    mAP@0.5, mAP@0.5:0.95, AP@0.5 per class
    results : list of {'scores','labels','boxes'(xyxy pixel)} — postprocessors 출력
    targets : list of {'boxes'(cx,cy,w,h norm),'labels'}
    """
    if iou_thrs is None:
        iou_thrs = np.arange(0.5, 1.0, 0.05)   # 10 thresholds

    # 예측/GT 수집 (클래스별)
    preds_by_cls = {c: [] for c in range(n_cls)}   # (score, img_i, box_xyxy)
    gts_by_cls   = {c: [] for c in range(n_cls)}   # (img_i, box_xyxy)

    for img_i, (res, tgt) in enumerate(zip(results, targets)):
        scores = res['scores'].cpu().numpy()
        labels = res['labels'].cpu().numpy()
        boxes  = res['boxes'].cpu().numpy()          # xyxy pixel

        for s, l, b in zip(scores, labels, boxes):
            if 0 <= l < n_cls:
                preds_by_cls[int(l)].append((float(s), img_i, b))

        gt_boxes  = box_cxcywh_to_xyxy_pixel(
            tgt['boxes'].cpu(), img_size).numpy()
        gt_labels = tgt['labels'].cpu().numpy()
        for l, b in zip(gt_labels, gt_boxes):
            if 0 <= l < n_cls:
                gts_by_cls[int(l)].append((img_i, b))

    ap_mat = np.zeros((n_cls, len(iou_thrs)))   # (class, iou_thr)

    for c in range(n_cls):
        preds = sorted(preds_by_cls[c], key=lambda x: x[0], reverse=True)
        gts   = gts_by_cls[c]
        n_gt  = len(gts)
        if n_gt == 0:
            continue

        # GT를 이미지별로 정리
        gt_img = {}
        for img_i, box in gts:
            gt_img.setdefault(img_i, []).append(box)

        for t_idx, iou_thr in enumerate(iou_thrs):
            tp = np.zeros(len(preds))
            fp = np.zeros(len(preds))
            matched = {i: [False]*len(b) for i, b in gt_img.items()}

            for p_idx, (_, img_i, pred_box) in enumerate(preds):
                if img_i not in gt_img:
                    fp[p_idx] = 1; continue
                gt_arr  = np.array(gt_img[img_i])
                ious    = box_iou_np(pred_box, gt_arr)
                best_j  = int(ious.argmax())
                if ious[best_j] >= iou_thr and not matched[img_i][best_j]:
                    tp[p_idx] = 1
                    matched[img_i][best_j] = True
                else:
                    fp[p_idx] = 1

            cum_tp  = np.cumsum(tp)
            cum_fp  = np.cumsum(fp)
            rec     = cum_tp / (n_gt + 1e-8)
            prec    = cum_tp / (cum_tp + cum_fp + 1e-8)
            ap_mat[c, t_idx] = compute_ap(rec, prec)

    map50      = float(ap_mat[:, 0].mean())        # mAP@IoU=0.50
    map50_95   = float(ap_mat.mean())              # mAP@IoU=0.50:0.95
    ap50_cls   = ap_mat[:, 0]                       # AP@0.5 per class
    return map50, map50_95, ap50_cls


# ── checkpoint 로드 (재개 시 주석 해제) ─────────────────────────
# ckpt_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(ckpt_path):
#     ckpt = torch.load(ckpt_path, map_location=device)
#     model.load_state_dict(ckpt['model_state_dict'])
#     optimizer.load_state_dict(ckpt['optimizer_state_dict'])
#     scaler.load_state_dict(ckpt['scaler_state_dict'])
#     if ema_model and 'ema_state_dict' in ckpt:
#         ema_model.module.load_state_dict(ckpt['ema_state_dict'])
#     if 'scheduler_state_dict' in ckpt:
#         scheduler.load_state_dict(ckpt['scheduler_state_dict'])
#     start_epoch = ckpt['epoch'] + 1
#     best_val_map = ckpt.get('best_val_map', 0.0)
#     print(f"✅ Resumed from epoch {start_epoch}")


# ── 학습 루프 ─────────────────────────────────────────────────────
print("\n" + "="*80)
print(f"🚀 Starting DINO Training")
print("="*80)

weight_dict  = criterion.weight_dict
target_sizes = torch.tensor([[input_size, input_size]], device=device)

for epoch in range(start_epoch, cfg.epochs):

    # ── Train ──────────────────────────────────────────────────────
    model.train(); criterion.train()
    e_loss = e_cls = e_bbox = e_giou = 0.0

    train_pbar = tqdm(train_loader, total=len(train_loader),
                      desc=f'Epoch {epoch+1}/{cfg.epochs} [Train]')

    for images, targets in train_pbar:
        samples = nested_tensor_from_tensor_list(list(images)).to(device)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs   = model(samples, targets)
            loss_dict = criterion(outputs, targets)
            losses    = sum(loss_dict[k] * weight_dict[k]
                           for k in loss_dict if k in weight_dict)

        optimizer.zero_grad()
        scaler.scale(losses).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_max_norm)
        scaler.step(optimizer); scaler.update()

        if ema_model is not None:
            ema_model.update(model)

        e_loss  += losses.item()
        e_cls   += loss_dict.get('loss_ce',   torch.tensor(0.)).item()
        e_bbox  += loss_dict.get('loss_bbox', torch.tensor(0.)).item()
        e_giou  += loss_dict.get('loss_giou', torch.tensor(0.)).item()

        mem = f'{torch.cuda.memory_reserved()/1e9:.2f}G' if torch.cuda.is_available() else '-'
        n_done = train_pbar.n + 1
        train_pbar.set_postfix({
            'loss': f'{e_loss/n_done:.4f}',
            'cls':  f'{e_cls/n_done:.4f}',
            'bbox': f'{e_bbox/n_done:.4f}',
            'mem':  mem,
        })

    n = len(train_loader)
    avg_loss = e_loss/n; avg_cls = e_cls/n; avg_bbox = e_bbox/n; avg_giou = e_giou/n
    train_losses.append(avg_loss)

    # ── Validation ─────────────────────────────────────────────────
    eval_model = ema_model.module if ema_model is not None else model
    eval_model.eval(); criterion.eval()

    all_results, all_tgt_val = [], []
    val_pbar = tqdm(val_loader, total=len(val_loader),
                    desc=f'Epoch {epoch+1}/{cfg.epochs} [Val]')

    with torch.no_grad():
        for images, targets in val_pbar:
            samples = nested_tensor_from_tensor_list(list(images)).to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = eval_model(samples, targets)

            tgt_sizes = target_sizes.repeat(len(targets), 1)
            all_results.extend(postprocessors['bbox'](outputs, tgt_sizes))
            all_tgt_val.extend(targets)

    map50, map50_95, ap50_cls = compute_map(
        all_results, all_tgt_val, input_size, num_classes
    )
    val_map50s.append(map50)
    val_map5095s.append(map50_95)

    # ── LR Scheduler step ─────────────────────────────────────────
    scheduler.step()

    # ── 로그 출력 ─────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{cfg.epochs}")
    print(f"  Train  — Loss: {avg_loss:.4f}  "
          f"(cls: {avg_cls:.4f} | bbox: {avg_bbox:.4f} | giou: {avg_giou:.4f})")
    print(f"  Val    — mAP@0.5: {map50:.4f}   mAP@0.5:0.95: {map50_95:.4f}")
    print(f"  AP@0.5 per class:")
    for i, ap in enumerate(ap50_cls):
        print(f"    [{i}] {class_names[i]:20s}: {ap:.4f}")
    print(f"  LR     — transformer: {optimizer.param_groups[0]['lr']:.2e}  "
          f"backbone: {optimizer.param_groups[1]['lr']:.2e}")
    print(f"{'='*80}\n")

    # ── 체크포인트 ────────────────────────────────────────────────
    ckpt = {
        'epoch':                  epoch,
        'model_state_dict':       model.state_dict(),
        'optimizer_state_dict':   optimizer.state_dict(),
        'scheduler_state_dict':   scheduler.state_dict(),
        'scaler_state_dict':      scaler.state_dict(),
        'train_loss':             avg_loss,
        'val_map50':              map50,
        'val_map50_95':           map50_95,
        'best_val_map':           best_val_map,
    }
    if ema_model is not None:
        ckpt['ema_state_dict'] = ema_model.module.state_dict()
    torch.save(ckpt, os.path.join(save_dir, 'last_model.pt'))

    if map50 > best_val_map:
        best_val_map = map50
        torch.save(ckpt, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 New best model!  mAP@0.5: {map50:.4f}\n")

    # ── 시각화 (10 epoch마다) ─────────────────────────────────────
    if (epoch + 1) % 10 == 0:
        vis_idx   = random.randint(0, len(val_dataset)-1)
        img_t, boxes_t, labels_t = val_dataset[vis_idx]
        colors = ['red','limegreen','yellow','magenta','dodgerblue','orange']

        with torch.no_grad():
            samp = nested_tensor_from_tensor_list([img_t]).to(device)
            tgt  = [{'boxes': boxes_t.to(device), 'labels': labels_t.to(device)}]
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = eval_model(samp, tgt)
            res = postprocessors['bbox'](out, target_sizes)[0]

        keep        = res['scores'] > 0.3   # 0.5 → 0.3: 소수 클래스도 시각화
        pred_boxes  = res['boxes'][keep].cpu().numpy()
        pred_labels = res['labels'][keep].cpu().numpy()
        pred_scores = res['scores'][keep].cpu().numpy()
        gt_boxes_px = box_cxcywh_to_xyxy_pixel(boxes_t, input_size).numpy()

        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        for ax, bbs, lbls, title in [
            (axes[0], gt_boxes_px,  labels_t.numpy(), f'GT ({len(labels_t)})'),
            (axes[1], pred_boxes,   pred_labels,       f'Pred thr=0.3 ({len(pred_labels)})'),
        ]:
            ax.imshow(img_t.numpy().transpose(1,2,0))
            for j, ((x1,y1,x2,y2), cls) in enumerate(zip(bbs, lbls)):
                ax.add_patch(patches.Rectangle(
                    (x1,y1), x2-x1, y2-y1,
                    linewidth=1.5, edgecolor=colors[int(cls)%len(colors)], facecolor='none'))
                if ax is axes[1]:   # pred에만 score 표시
                    ax.text(x1, y1-2, f'{pred_scores[j]:.2f}',
                            color=colors[int(cls)%len(colors)], fontsize=6)
            ax.set_title(title, fontsize=12); ax.axis('off')
        # 범례
        legend_elements = [
            patches.Patch(color=colors[i], label=class_names[i])
            for i in range(num_classes)
        ]
        axes[0].legend(handles=legend_elements, loc='upper right', fontsize=7, framealpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'pred_epoch_{epoch+1}.png'), dpi=150, bbox_inches='tight')
        plt.close()
        print(f"📸 pred_epoch_{epoch+1}.png\n")

    # ── 학습 곡선 (100 epoch마다) ──────────────────────────────────
    if (epoch + 1) % 100 == 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(train_losses, 'b-'); axes[0].set_title('Train Loss'); axes[0].grid(True)
        axes[1].plot(val_map50s,   'r-', label='mAP@0.5')
        axes[1].plot(val_map5095s, 'b-', label='mAP@0.5:0.95')
        axes[1].set_title('Val mAP'); axes[1].legend(); axes[1].grid(True)
        for ax in axes: ax.set_xlabel('Epoch')
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'progress_epoch_{epoch+1}.png'), dpi=150)
        plt.close()
        print(f"📊 progress_epoch_{epoch+1}.png\n")

print("\n" + "="*80)
print("🎯 Training Complete!")
print(f"  Best mAP@0.5 : {best_val_map:.4f}")
print(f"  Models saved : {save_dir}")
print("="*80)
